# Eksploracja danych GIOŚ

## Importy i dostęp do kodu projektu

In [ ]:
import sys

sys.path.append("../src")

In [ ]:
from smogcast.ingest.gios import (
    get_all_stations,
    get_archival_data_by_sensor,
)

In [ ]:
from smogcast.ingest.gios import generate_date_ranges

## Pobranie stacji GIOŚ

In [ ]:
stations = get_all_stations()

In [ ]:
len(stations)

In [ ]:
stations[0]

In [ ]:
from smogcast.processing.filters import filter_silesian_stations

In [ ]:
silesian_stations = filter_silesian_stations(stations)

In [ ]:
len(silesian_stations)

In [ ]:
[
    (
        station["Identyfikator stacji"],
        station["Nazwa stacji"],
        station["Nazwa miasta"],
    )
    for station in silesian_stations
]

## Test na jednej stacji

## Zapis surowych danych

In [ ]:
import sys

sys.path.append("../src")

In [ ]:
from smogcast.ingest.gios import (
    get_all_stations,
    get_archival_data_by_sensor,
    get_data_for_sensors,
)

In [ ]:
from smogcast.ingest.gios import generate_date_ranges

In [ ]:
from smogcast.processing.filters import filter_silesian_stations

In [ ]:
import pandas as pd

In [ ]:
stations = get_all_stations()

In [ ]:
silesian_stations = filter_silesian_stations(stations)

In [ ]:
station_id = 814

In [ ]:
import httpx

In [ ]:
url = f"https://api.gios.gov.pl/pjp-api/v1/rest/station/sensors/{station_id}"

response = httpx.get(url, timeout=30.0)
response.raise_for_status()

sensor_data = response.json()

In [ ]:
sensors = sensor_data["Lista stanowisk pomiarowych dla podanej stacji"]

In [ ]:
pm_sensors = [
    sensor
    for sensor in sensors
    if sensor.get("Wskaźnik") in {
        "pył zawieszony PM10",
        "pył zawieszony PM2.5",
    }
]

In [ ]:
sensor_ids = [
    sensor["Identyfikator stanowiska"]
    for sensor in pm_sensors
]

In [ ]:
sensor_ids

In [ ]:
all_measurements, failed_ranges = get_data_for_sensors(
    sensor_ids,
    "2026-08-20 10:00",
    "2026-08-20 12:00",
    days_per_range=1,
)

In [ ]:
df = pd.DataFrame(all_measurements)

In [ ]:
df

In [ ]:
df.to_parquet(
    "../data/raw/gios_pm_measurements.parquet",
    index=False,
)

In [ ]:
pd.read_parquet("../data/raw/gios_pm_measurements.parquet").head()

## Lista slaskich stacji z pm10 albo pm2.5

In [ ]:
stations = get_all_stations()

In [ ]:
silesian_stations = filter_silesian_stations(stations)

In [ ]:
import httpx
import time

In [ ]:
stations_with_pm = []

In [ ]:
for station in silesian_stations:
    station_id = station["Identyfikator stacji"]

    url = (
        "https://api.gios.gov.pl/pjp-api/"
        f"v1/rest/station/sensors/{station_id}"
    )

    response = httpx.get(url, timeout=30.0)
    response.raise_for_status()

    sensor_data = response.json()

    sensors = sensor_data[
        "Lista stanowisk pomiarowych dla podanej stacji"
    ]

    pm_sensors = [
        sensor
        for sensor in sensors
        if sensor.get("Wskaźnik") in {
            "pył zawieszony PM10",
            "pył zawieszony PM2.5",
        }
    ]

    if pm_sensors:
        stations_with_pm.append(
            {
                "station_id": station_id,
                "station_name": station["Nazwa stacji"],
                "city": station["Nazwa miasta"],
                "pm_sensors": pm_sensors,
            }
        )

    time.sleep(1)

In [ ]:
len(stations_with_pm)

In [ ]:
import pandas as pd

In [ ]:
stations_pm_df = pd.DataFrame(
    [
        {
            "station_id": station["station_id"],
            "station_name": station["station_name"],
            "city": station["city"],
            "pm10_count": sum(
                sensor.get("Wskaźnik") == "pył zawieszony PM10"
                for sensor in station["pm_sensors"]
            ),
            "pm25_count": sum(
                sensor.get("Wskaźnik") == "pył zawieszony PM2.5"
                for sensor in station["pm_sensors"]
            ),
        }
        for station in stations_with_pm
    ]
)

stations_pm_df

In [ ]:
len(stations_pm_df)

In [ ]:
stations_pm_df.to_parquet(
    "../data/raw/gios_silesian_pm_stations.parquet",
    index=False,
)

In [ ]:
stations_pm_df.to_csv(
    "../data/raw/gios_silesian_pm_stations.csv",
    index=False,
)

## Lista polskich stacji z pm10 albo pm2.5

In [ ]:
stations_with_pm_poland = []

In [ ]:
for station in stations:
    station_id = station["Identyfikator stacji"]

    url = (
        "https://api.gios.gov.pl/pjp-api/"
        f"v1/rest/station/sensors/{station_id}"
    )

    response = httpx.get(url, timeout=30.0)
    response.raise_for_status()

    sensor_data = response.json()

    sensors = sensor_data[
        "Lista stanowisk pomiarowych dla podanej stacji"
    ]

    pm_sensors = [
        sensor
        for sensor in sensors
        if sensor.get("Wskaźnik") in {
            "pył zawieszony PM10",
            "pył zawieszony PM2.5",
        }
    ]

    if pm_sensors:
        stations_with_pm_poland.append(
            {
                "station_id": station_id,
                "station_name": station["Nazwa stacji"],
                "city": station["Nazwa miasta"],
                "voivodeship": station["Województwo"],
                "pm_sensors": pm_sensors,
            }
        )

    time.sleep(1)

In [ ]:
len(stations_with_pm_poland)

In [ ]:
stations_pm_poland_df = pd.DataFrame(
    [
        {
            "station_id": station["station_id"],
            "station_name": station["station_name"],
            "city": station["city"],
            "voivodeship": station["voivodeship"],
            "pm10_count": sum(
                sensor.get("Wskaźnik") == "pył zawieszony PM10"
                for sensor in station["pm_sensors"]
            ),
            "pm25_count": sum(
                sensor.get("Wskaźnik") == "pył zawieszony PM2.5"
                for sensor in station["pm_sensors"]
            ),
        }
        for station in stations_with_pm_poland
    ]
)

In [ ]:
stations_pm_poland_df

In [ ]:
stations_pm_poland_df["voivodeship"].value_counts()

In [ ]:
stations_pm_poland_df.to_parquet(
    "../data/raw/gios_poland_pm_stations.parquet",
    index=False,
)

In [ ]:
stations_pm_poland_df.to_csv(
    "../data/raw/gios_poland_pm_stations.csv",
    index=False,
)

### sprawdzenie zakresów dat i ewentualnych luk

In [ ]:
df["Data"] = pd.to_datetime(df["Data"])

In [ ]:
min_date = df["Data"].min()
max_date = df["Data"].max()

print("Najstarszy pomiar:", min_date)
print("Najnowszy pomiar:", max_date)

## Lista slaskich stacji- 3 lata

In [ ]:
sensor_records = []

for station in stations_with_pm:
    for sensor in station["pm_sensors"]:
        indicator = sensor.get("Wskaźnik")

        if indicator == "pył zawieszony PM10":
            parameter = "PM10"
        elif indicator == "pył zawieszony PM2.5":
            parameter = "PM2.5"
        else:
            continue

        sensor_records.append(
            {
                "station_id": station["station_id"],
                "sensor_id": sensor["Identyfikator stanowiska"],
                "parameter": parameter,
            }
        )

In [ ]:
sensor_records[:10]

In [ ]:
len(sensor_records)

In [ ]:
date_from = "2023-08-20 00:00"
date_to = "2026-08-20 23:00"

In [ ]:
from smogcast.ingest.gios import get_data_for_sensor_records

In [ ]:
all_measurements, failed_ranges = get_data_for_sensor_records(
    sensor_records,
    date_from,
    date_to,
    days_per_range=30,
)

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent / "src"))

from smogcast.ingest.gios import get_data_for_sensor_records

In [ ]:
CURRENT_GIOS_PATH = "../data/raw/gios_pm_measurements_1y.parquet"
OLDER_GIOS_PATH = "../data/raw/gios_pm_measurements_older.parquet"
FULL_GIOS_PATH = "../data/raw/gios_pm_measurements_3y.parquet"

DATE_FROM = "2023-08-20 00:00"
DATE_TO = "2025-08-19 23:00"

In [ ]:
current_gios = pd.read_parquet(CURRENT_GIOS_PATH)

current_gios.head()

In [ ]:
current_gios.shape

In [ ]:
current_gios.columns.tolist()

In [ ]:
sensor_records_df = (
    current_gios[
        [
            "station_id",
            "sensor_id",
            "station_name",
            "city",
            "voivodeship",
            "Parametr",
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "Parametr": "parameter",
        }
    )
    .reset_index(drop=True)
)

sensor_records = sensor_records_df.to_dict(
    orient="records"
)

print("Liczba sensorów:", len(sensor_records))

sensor_records_df.head()

In [ ]:
sensor_records_df["station_id"].nunique()

In [ ]:
import contextlib
import io

from tqdm.auto import tqdm

In [ ]:
older_measurements = []
failed_ranges = []

for sensor in tqdm(
    sensor_records,
    desc="Pobieranie danych GIOŚ",
    unit="sensor",
):
    # Wycisza komunikaty print() z funkcji ingestu.
    with contextlib.redirect_stdout(io.StringIO()):
        measurements, sensor_failed_ranges = get_data_for_sensor_records(
            sensor_records=[sensor],
            date_from=DATE_FROM,
            date_to=DATE_TO,
            days_per_range=30,
        )

    older_measurements.extend(measurements)
    failed_ranges.extend(sensor_failed_ranges)

In [ ]:
older_gios = pd.DataFrame(older_measurements)

older_gios.shape

In [ ]:
older_gios["Data"] = pd.to_datetime(older_gios["Data"])

print(
    older_gios["Data"].min(),
    "→",
    older_gios["Data"].max(),
)

In [ ]:
print("Nieudane zakresy:", len(failed_ranges))

In [ ]:
older_gios.to_parquet(
    "../data/raw/gios_pm_measurements_older.parquet",
    index=False,
)

In [ ]:
current_gios = pd.read_parquet(
    "../data/raw/gios_pm_measurements_1y.parquet"
)

In [ ]:
current_gios["Data"] = pd.to_datetime(current_gios["Data"])

In [ ]:
gios_3y = pd.concat(
    [
        older_gios,
        current_gios,
    ],
    ignore_index=True,
)

In [ ]:
before = len(gios_3y)

gios_3y = (
    gios_3y
    .drop_duplicates(
        subset=["sensor_id", "Data"]
    )
    .reset_index(drop=True)
)

after = len(gios_3y)

print("Przed:", before)
print("Po:", after)
print("Duplikaty usunięte:", before - after)

In [ ]:
print(
    "Zakres:",
    gios_3y["Data"].min(),
    "→",
    gios_3y["Data"].max(),
)

print("Rekordów:", len(gios_3y))
print("Stacji:", gios_3y["station_id"].nunique())
print("Sensorów:", gios_3y["sensor_id"].nunique())

In [ ]:
gios_3y.to_parquet(
    "../data/raw/gios_pm_measurements_3y.parquet",
    index=False,
)